### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

##### Some Build-in MiddleWare

* **Summarization**
* **Human in the Loop**
* **Model Call Limit**
* **Tool Call Limit**
* **Model Feedback**
* **PII Detection** ...

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

#### Summarization MiddeWare
---------

**LangChain's `SummarizationMiddleware` automatically compresses long conversation histories by summarizing older messages once a specific token or message threshold is reached.** 

* How It Works
    - Monitors Usage: Tracks the conversation length using token counts, message counts, or a fraction of the total model context window.
    - Triggers Compression: Activates right before calling the language model when limits are approached.
    - Retains Context: Replaces older messages with a compact summary while keeping a specified number of recent messages intact.
    - Preserves Pairing: Keeps AI and tool message pairs together to maintain logical continuity.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver 
from langchain_core.messages import HumanMessage, SystemMessage

## Message based summarization
agent = create_agent(
    model="gpt-5-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='gpt-4o-mini',
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ] 
)

In [3]:
## Run with thread_id
config = {'configurable':{'thread_id':'test-1'}}

## test data
questions = [
    "what is 2+2",
    "what is 2/3",
    "what is 45*2",
    "what is 100/4",
    "what is 3*5",
    "what is 7-4",
    "what is 2^2"
]

for q in questions:
    response = agent.invoke(
        {'messages':[HumanMessage(content=q)]},config   
    )
    print(f"Messages: {response}")
    print(f"Messasges: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='what is 2+2', additional_kwargs={}, response_metadata={}, id='905b2132-43ff-4157-8fcb-ab031ab7e18b'), AIMessage(content='2 + 2 = 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 12, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EKSs8V8geFH6cntiOsDwJS0IwuG2P', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a06da9-fac6-7bb1-975f-8dd35c5c4a6f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 17, 'total_tokens': 29, 'i

#### Summarization based on token size

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:

    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi."""


agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ]
)

In [5]:
config = {'configurable':{'thread_id':'test-1'}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(
        len(str(m.content))
        for m in messages
    )
    return total_chars // 4  # ~4 chars = 1 token

In [6]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {
            "messages": [HumanMessage(content=f"Find hotels in {city}")]
        },
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

Paris: ~139 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='90298eb0-6518-4925-a3eb-71b68110acc5'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_79b520a473', 'id': 'chatcmpl-EKSsNyMWTMD2PBWgOpoT0dTsy6uBf', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a06daa-377b-7000-be4e-36e985418af5-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_HHSSklBiP7YCrYH

#### Based on Fraction

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:

    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi."""


agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction", 0.005),  # 0.5% ~640 tokens
            keep=("tokens", 200)          # 0.2% ~256 tokens
        )
    ]
)

In [8]:
config = {'configurable':{'thread_id':'test-1'}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(
        len(str(m.content))
        for m in messages
    )
    return total_chars // 4  # ~4 chars = 1 token

In [9]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {
            "messages": [HumanMessage(content=f"Find hotels in {city}")]
        },
        config=config
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000
    print(f"{city}: ~{tokens} tokens, ({fraction:.4%}), {len(response['messages'])} messages")
    print(f"{response['messages']}")

Paris: ~155 tokens, (0.1211%), 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='098159b3-aeb0-410d-bf21-159d3c7194ca'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_79b520a473', 'id': 'chatcmpl-EKSsk8QPY8jTqT24czYVVomarVXW4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a06daa-92af-7a73-82ff-10134c965cb2-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_nvhc

#### Human in the Loop Middleware
----

**Human-in-the-loop (HITL) middleware is an agent control layer that intercepts AI tool calls to pause execution and require human review before sensitive or risky actions run.** 

* How It Works
    - Interception: The middleware monitors tool calls generated by the language model during its reasoning loop.
    - Policy Matching: It checks proposed actions against defined rules (such as requiring approval only for specific tools like send_email or execute_sql).
    - State Persistence: When triggered, it halts execution and saves the current agent state using a checkpointer.
    - Human Decisions: A human reviewer can approve the action, edit the parameters, reject it with feedback, or respond directly.
    - Resumption: The agent safely picks up execution from the exact interruption point based on the human's input

In [10]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its email id"""
    return f'Email content for id: {email_id}'

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f'Email sent to {recipient} with subject "{subject}"'

In [15]:
agent = create_agent(
    model='gpt-4o-mini',
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [16]:
config = {'configurable': {'thread_id':'test_approve'}}

## Step 1: request
result = agent.invoke(
    {'messages': [HumanMessage(content="send an email to john@test.com with subject 'Hello' and body 'How are you?")]},
    config=config
)

In [17]:
result

{'messages': [HumanMessage(content="send an email to john@test.com with subject 'Hello' and body 'How are you?", additional_kwargs={}, response_metadata={}, id='fb87bee6-13b0-4d57-88b3-22374ecfa23e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 99, 'total_tokens': 127, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_62666fb687', 'id': 'chatcmpl-EKStu7NXyhxMbzgpoBze4qGEe7nQ0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a06dab-a8b3-78a2-97f6-199d1b631c4a-0', tool_calls=[{'name': 'send_email_tool', 'args': {'rec

In [18]:
# Step 2: Approve
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "approve"
                }
            ]
        }
    ),
    config=config
)

print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been successfully sent to john@test.com with the subject "Hello" and the body "How are you?".


In [19]:
result

{'messages': [HumanMessage(content="send an email to john@test.com with subject 'Hello' and body 'How are you?", additional_kwargs={}, response_metadata={}, id='fb87bee6-13b0-4d57-88b3-22374ecfa23e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 99, 'total_tokens': 127, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_62666fb687', 'id': 'chatcmpl-EKStu7NXyhxMbzgpoBze4qGEe7nQ0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a06dab-a8b3-78a2-97f6-199d1b631c4a-0', tool_calls=[{'name': 'send_email_tool', 'args': {'rec

#### Reject

In [21]:
agent = create_agent(
    model='gpt-4o-mini',
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [22]:
config = {'configurable': {'thread_id':'test_reject'}}

## Step 1: request
result = agent.invoke(
    {'messages': [HumanMessage(content="send an email to john@test.com with subject 'Hello' and body 'How are you?")]},
    config=config
)

In [23]:
# Step 2: Approve
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject"
                }
            ]
        }
    ),
    config=config
)

print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: It appears that the email sending request was not executed. If you would like me to proceed with sending the email, please confirm, and I'll take care of it for you.


### Edit

In [24]:
agent = create_agent(
    model='gpt-4o-mini',
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [25]:
config = {'configurable': {'thread_id':'test_reject'}}

## Step 1: request
result = agent.invoke(
    {'messages': [HumanMessage(content="send an email to wrong@test.com with subject 'Hello' and body 'How are you?")]},
    config=config
)

In [26]:
# Step 2: Edit
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email_tool",  # Tool name
                        "args": {
                            "recipient": "correct@mail.com",
                            "subject": "corrected_subject",
                            "body": "This was edited by human before sending..."
                        }
                    }
                }
            ]
        }
    ),
    config=config
)

print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email was sent successfully to correct@mail.com with the subject "corrected_subject".


In [27]:
result

{'messages': [HumanMessage(content="send an email to wrong@test.com with subject 'Hello' and body 'How are you?", additional_kwargs={}, response_metadata={}, id='ff2a79a2-e5be-4115-8d1e-60cf0bc3d4b4'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 99, 'total_tokens': 127, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_62666fb687', 'id': 'chatcmpl-EKSyEKP7FcpA36kxc0qsQdpzIYRmh', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a06daf-c058-72e1-b78b-5e8bf1d2e45d-0', tool_calls=[{'type': 'tool_call', 'name': 'send_emai